## Part 2: PepMLM — Peptide Generation with the Pre-trained Model

Uses `TianlaiChen/PepMLM-650M`, the ESM-2 650M model fine-tuned on peptide-protein pairs  
from the Nature paper. We load it directly and use it to **generate new peptide sequences**  
for receptors in our test set.

**Pipeline:**
1. Load pretrained PepMLM-650M  
2. For each receptor, mask a C-terminal peptide region  
3. Decode the masked positions → generated peptide  
4. Compute perplexity (PPL) to score each generated sequence

### 2.1 — Imports

In [7]:
import math
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


### 2.2 — Load Data

In [8]:
DATA = "../data/"

test_df = pd.read_csv(DATA + "pepnn_test_dataset.csv")
test_df = test_df.dropna(subset=["Receptor Sequence", "Sequence"]).reset_index(drop=True)
test_df["pep_len"] = test_df["Sequence"].str.len()

print("Test samples:", len(test_df))
test_df[["PDB", "Receptor Sequence", "Sequence", "pep_len"]].head()

Test samples: 92


,PDB,Receptor Sequence,Sequence,pep_len
0,6VME,QSENNDIDEVIIPTAPLYKQILNLYAEENAIEDTIFYLGEALRRGV...,NASSLYGISAMDGVPFTLH,19
1,6TYT,MHHHHHHMARAAKSAVVLCMDVGLAMSHSNQGKESPFEQAKKVMML...,AKGLFM,6
2,6TYT,MHHHHHHMARAAKSAVVLCMDVGLAMSHSNQGKESPFEQAKKVMML...,RKRILPTWMLA,11
3,7MGV,MGSSHHHHHHSSGLVPRGSHMNKILIITHTADNFSIDKVTEYIDKN...,TLKYPSDSDEG,11
4,6RMV,MSELDQLRQEAEQLKNQIRDARKACADATLSQITNNIDPVGRIQMR...,KRPKALKLLGMED,13


### 2.3 — Load PepMLM-650M

> **Note:** This model is 650M parameters (~2.5 GB). First load will download weights from HuggingFace. Inference runs on CPU but will be slow — a few seconds per sequence.

In [9]:
MODEL_NAME = "TianlaiChen/PepMLM-650M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
model     = model.to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters  : {total_params:,}")

Loading weights: 100%|██████████| 540/540 [00:00<00:00, 54688.50it/s]

Model loaded: TianlaiChen/PepMLM-650M
Parameters  : 651,043,254


### 2.4 — Generation & Scoring Functions

**Generation:** Append `pep_len` mask tokens to the receptor, run one forward pass, take argmax at each masked position.

**Perplexity (PPL):** Mask each peptide position one at a time, record the model's log probability of the correct token, average across all positions.  
`PPL = exp(mean negative log-likelihood)` — lower is better.

In [10]:
MAX_LEN = 512

def generate_peptide(receptor_seq: str, pep_len: int) -> str:
    """Mask pep_len positions at the C-terminus of receptor and decode."""
    masked_input = receptor_seq + tokenizer.mask_token * pep_len
    inputs = tokenizer(masked_input, return_tensors="pt",
                       truncation=True, max_length=MAX_LEN).to(device)

    mask_positions = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_ids = logits[0, mask_positions].argmax(dim=-1)
    return tokenizer.decode(predicted_ids, skip_special_tokens=True).replace(" ", "")


def compute_ppl(receptor_seq: str, peptide_seq: str) -> float:
    """Pseudo-perplexity: mask each peptide position one at a time."""
    combined = receptor_seq + peptide_seq
    base_inputs = tokenizer(combined, return_tensors="pt",
                            truncation=True, max_length=MAX_LEN).to(device)

    pep_token_ids = tokenizer(peptide_seq, add_special_tokens=False)["input_ids"]
    n = len(pep_token_ids)

    # Find where the peptide starts in the tokenized combined sequence
    full_ids = base_inputs["input_ids"][0].tolist()
    receptor_token_ids = tokenizer(receptor_seq, add_special_tokens=False)["input_ids"]
    pep_start = 1 + len(receptor_token_ids)  # offset by [CLS]

    log_probs = []
    for i in range(n):
        pos = pep_start + i
        if pos >= base_inputs["input_ids"].shape[1]:
            break
        masked = base_inputs["input_ids"].clone()
        masked[0, pos] = tokenizer.mask_token_id

        with torch.no_grad():
            logits = model(input_ids=masked,
                           attention_mask=base_inputs["attention_mask"]).logits

        probs = F.softmax(logits[0, pos], dim=-1)
        true_id = pep_token_ids[i]
        log_probs.append(torch.log(probs[true_id] + 1e-10).item())

    return math.exp(-sum(log_probs) / len(log_probs)) if log_probs else float("inf")


print("Functions defined.")

Functions defined.


### 2.5 — Generate Peptides for Test Set

We run on the first 10 test receptors to keep runtime reasonable on CPU.

In [ ]:
subset = test_df.head(10).copy()

results = []
for _, row in subset.iterrows():
    receptor    = row["Receptor Sequence"]
    true_pep    = row["Sequence"]
    pep_len     = row["pep_len"]

    generated   = generate_peptide(receptor, pep_len)
    ppl_gen     = compute_ppl(receptor, generated)
    ppl_true    = compute_ppl(receptor, true_pep)

    results.append({
        "PDB":           row["PDB"],
        "True peptide":  true_pep,
        "Generated":     generated,
        "PPL (true)":    round(ppl_true,  3),
        "PPL (gen)":     round(ppl_gen,   3),
    })
    print(f"{row['PDB']:6s}  true={true_pep:25s}  gen={generated:25s}  PPL_true={ppl_true:.2f}  PPL_gen={ppl_gen:.2f}")

results_df = pd.DataFrame(results)
results_df